In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Download stopwords
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

# Load dataset
df = pd.read_csv("harvard_dataset_data_df.csv")

# Check columns
print(df.columns)

# Change 'abstract' to your text column name
text_col = "abstract"
df[text_col] = df[text_col].fillna("").astype(str)

# NLP cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    words = [w for w in text.split() if w not in stop_words]
    return " ".join(words)

df["clean_text"] = df[text_col].apply(clean_text)

# TF-IDF
vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(df["clean_text"])

# K-Means
k = 5
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = model.fit_predict(X)

# Top words in each cluster
words = vectorizer.get_feature_names_out()
centers = model.cluster_centers_.argsort()[:, ::-1]

for i in range(k):
    print(f"Cluster {i}:",
          ", ".join(words[x] for x in centers[i, :10]))

# PCA visualization
pca = PCA(n_components=2)
X2 = pca.fit_transform(X.toarray())

plt.scatter(X2[:, 0], X2[:, 1], c=df["cluster"])
plt.title("K-Means NLP Clusters")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()

# Save results
df.to_csv("harvard_kmeans_results.csv", index=False)

print(df[[text_col, "cluster"]].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Index(['Unnamed: 0', 'data_product', 'file_name', 'str_col_summary',
       'num_full_na_cols', 'num_cols', 'd_name', 'd_description'],
      dtype='str')


KeyError: 'abstract'